In [0]:
-- Schema-on-Read" (No SQL)
-- Consultamos las imágenes como una colección de documentos binarios
SELECT 
  path, 
  length / 1024 as size_kb,
  modificationTime,
  -- Extraemos el ID del archivo para usarlo como 'Primary Key' virtual
  regexp_extract(path, '[^/]+$', 0) as image_key
FROM binaryFile.`/Volumes/workspace/bronze/yeik/plants/train_images`
LIMIT 10;

-- NoSQL: Flattening
-- Convertimos el CSV en una estructura dinámica
-- Usamos la sintaxis de opciones para forzar la lectura de los nombres de columnas
-- Usamos read_files para cargar el archivo crudo y darle estructura al vuelo
SELECT 
  image as plant_id,
  explode(split(labels, ' ')) as disease_tag
FROM read_files(
  '/Volumes/workspace/bronze/yeik/plants/train.csv',
  format => 'csv',
  header => true
)
WHERE image IS NOT NULL;


-- Create a metadata table for the images
CREATE OR REPLACE TABLE workspace.bronze.plants_metadata AS
SELECT 
  -- Corregido: modificationTime (CamelCase)
  img.path, 
  img.modificationTime, 
  img.length AS size_bytes,
  -- Extraemos el ID del archivo (Primary Key virtual)
  regexp_extract(img.path, '[^/]+$', 0) AS image_id,
  -- Traemos las etiquetas del CSV usando read_files
  meta.labels AS disease_category
FROM binaryFile.`/Volumes/workspace/bronze/yeik/plants/train_images` AS img
LEFT JOIN read_files(
  '/Volumes/workspace/bronze/yeik/plants/train.csv',
  format => 'csv',
  header => true
) AS meta
  ON regexp_extract(img.path, '[^/]+$', 0) = meta.image
WHERE img.length > 0;

CREATE OR REPLACE TABLE workspace.silver.plant_images_cleaned AS
SELECT 
  path,
  -- Limpiamos el nombre de la enfermedad (Capa Silver)
  disease_category AS disease_type,
  size_bytes AS file_size,
  modificationTime
FROM workspace.bronze.plants_metadata
-- Filtro de Calidad: Ignoramos archivos menores a 10KB (corruptos o miniaturas)
WHERE size_bytes > 10240 
  AND disease_category IS NOT NULL;

  -- Usamos un modelo servido para predecir enfermedades en nuevas imágenes
SELECT 
  path,
  -- Extraemos el nombre del archivo para referencia
  regexp_extract(path, '[^/]+$', 0) AS file_name,
  -- Inferencia de IA: Llamamos al modelo 'plant_disease_classifier'
  ai_query(
    "plant_disease_classifier", 
    request => struct(content) -- Spark maneja el binario automáticamente en el request
  ) AS model_prediction
FROM binaryFile.`/Volumes/workspace/bronze/yeik/plants/new_uploads/`
LIMIT 20;

-- Simulación de cómo se vería la salida de la IA si el modelo no está listo
SELECT 
  path,
  CASE 
    WHEN length > 500000 THEN 'Scab (Predicted)'
    WHEN length < 200000 THEN 'Healthy (Predicted)'
    ELSE 'Complex/Frog Eye Leaf Spot (Predicted)'
  END AS simulated_prediction
FROM binaryFile.`/Volumes/workspace/bronze/yeik/plants/new_uploads/`;